In [13]:
import yfinance as yf
import pandas as pd
import warnings

# Suppress warnings for cleaner output
warnings.filterwarnings("ignore")

# Exact mapping of your 78 firms to their PSX Tickers
company_ticker_map = {
    "Indus Dyeing & Manufacturing Ltd": "IDYM.KA",
    "Shahtaj Textile Ltd": "STJT.KA",
    "Azgard Nine Ltd": "ANL.KA",
    "Feroze 1888 Mills Ltd": "FML.KA",
    "Gul Ahmed Textile Mills Ltd": "GATM.KA",
    "Kohinoor Textile Mills Ltd": "KTML.KA",
    "Nishat (Chunian) Ltd": "NCL.KA",
    "Nishat Mills Ltd": "NML.KA",
    "Bannu Woollen Mills Ltd": "BNWM.KA",
    "Ibrahim Fibres Ltd": "IBFL.KA",
    "JDW Sugar Mills Ltd": "JDWS.KA",
    "Bestway Cement Ltd": "BWCL.KA",
    "Cherat Cement Company Ltd": "CHCC.KA",
    "D. G. Khan Cement Co. Ltd": "DGKC.KA",
    "Fauji Cement Company Ltd.": "FCCL.KA",
    "Kohat Cement Company Ltd.": "KOHC.KA",
    "Lucky Cement Ltd.": "LUCK.KA",
    "Maple Leaf Cement Factory Ltd": "MLCF.KA",
    "Pioneer Cement Ltd": "PIOC.KA",
    "Pakistan Tobacco Company Ltd.": "PAKT.KA",
    "Philip Morris (Pakistan) Ltd": "PMPK.KA",
    "Attock Refinery Ltd": "ATRL.KA",
    "Byco Petroleum Pakistan Ltd": "CNERGY.KA",  # Now Cnergyico PK Ltd
    "K-Electric Ltd": "KEL.KA",
    "Kot Addu Power Company Ltd": "KAPCO.KA",
    "Saif Power Ltd": "SPWL.KA",
    "The Hub Power Company Ltd": "HUBC.KA",
    "Nishat Chunian Power Ltd": "NCPL.KA",
    "Attock Petroleum Ltd": "APL.KA",
    "Hascol Petroleum Ltd": "HASCOL.KA",
    "Pakistan State Oil Company Ltd": "PSO.KA",
    "Shell Pakistan Ltd": "SHEL.KA",
    "Sui Northern Gas Pipelines Ltd.": "SNGP.KA",
    "Sui Southern Gas Co. Ltd": "SSGC.KA",
    "Mari Petroleum Company Ltd": "MARI.KA",
    "Oil & Gas Development Co. Ltd.": "OGDC.KA",
    "Pakistan Oilfields Ltd": "POL.KA",
    "Pakistan Petroleum Ltd": "PPL.KA",
    "International Industries Ltd": "INIL.KA",
    "International Steels Ltd": "ISL.KA",
    "Atlas Honda Ltd": "ATLH.KA",
    "Honda Atlas Cars (Pakistan) Ltd": "HCAR.KA",
    "Indus Motor Company Ltd": "INDU.KA",
    "Millat Tractors Ltd": "MTL.KA",
    "Pak Suzuki Motor Co. Ltd.": "PSMC.KA",
    "Agriauto Industries Ltd": "AGIL.KA",
    "Thal Ltd": "THALL.KA",
    "Pak Elektron Ltd": "PAEL.KA",
    "Pakistan International Bulk Term. Ltd.": "PIBTL.KA",
    "P. T. C. L. \"A\"": "PTC.KA",
    "Systems Ltd": "SYS.KA",
    "TRG Pakistan Ltd": "TRG.KA",
    "Engro Corporation Ltd": "ENGRO.KA",
    "Engro Fertilizers Ltd": "EFERT.KA",
    "Fatima Fertilizer Company Ltd": "FATIMA.KA",
    "Fauji Fertilizer Bin Qasim Ltd": "FFBL.KA",
    "Fauji Fertilizer Company Ltd": "FFC.KA",
    "Abbott Laboratories (Pak.) Ltd": "ABOT.KA",
    "AGP Ltd": "AGP.KA",
    "GlaxoSmithKline Pakistan Ltd": "GLAXO.KA",
    "GlaxoSmithKline Consumer HealthCare": "HALEON.KA", # Now Haleon Pakistan
    "The Searle Company Ltd": "SEARL.KA",
    "Archroma Pakistan Ltd": "ARPL.KA",
    "Colgate-Palmolive (Pakistan) Ltd": "COLG.KA",
    "Engro Polymer & Chemicals Ltd": "EPCL.KA",
    "I. C. I. Pakistan Ltd": "LCI.KA",             # Now Lucky Core Industries
    "Lotte Chemical Pakistan Ltd": "LOTCHEM.KA",
    "Packages Ltd": "PKGS.KA",
    "Unity Foods Ltd": "UNITY.KA",
    "Service Industries Ltd": "SRVI.KA",
    "Engro Foods Ltd": "FCEPL.KA",                # Now FrieslandCampina Engro
    "Murree Brewery Company Ltd": "MUREB.KA",
    "Nestle Pakistan Ltd": "NESTLE.KA",
    "National Foods Ltd": "NATF.KA",
    "Ghani Glass Ltd": "GHGL.KA",
    "Pakistan Services Ltd": "PSEL.KA",
    "Shifa International Hospitals Ltd": "SHFA.KA"
}

def main():
    start_date = "2020-01-01"
    end_date = "2024-12-31"
    manual_agp_file = "AGP_Manual_Data.csv" # File you download from PSX portal
    
    tickers = list(company_ticker_map.values())
    inv_map = {v: k for k, v in company_ticker_map.items()}

    print(f"Connecting to Yahoo Finance to download daily data for {len(tickers)} companies...")
    all_dataframes = []

    for i, ticker in enumerate(tickers, 1):
        company_name = inv_map[ticker]
        print(f"[{i}/{len(tickers)}] Fetching {ticker} ({company_name})...")
        
        try:
            stock = yf.Ticker(ticker)
            hist = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            # --- AGP DATA ADJUSTMENT ---
            if (hist.empty or ticker == "AGP.KA") and ticker == "AGP.KA":
                if os.path.exists(manual_agp_file):
                    print(f"   -> Yahoo missing AGP data. Loading from {manual_agp_file}...")
                    hist = pd.read_csv(manual_agp_file)
                    hist['Date'] = pd.to_datetime(hist['Date'])
                    # Ensure columns match yfinance format for consistency
                    hist = hist.rename(columns={'Price': 'Close', 'Open': 'Open', 'High': 'High', 'Low': 'Low', 'Vol.': 'Volume'})
                else:
                    print(f"   -> WARNING: No data for AGP. Please place '{manual_agp_file}' in this folder.")
                    continue

            if hist.empty:
                continue
                
            hist = hist.reset_index()
            if hist['Date'].dt.tz is not None:
                hist['Date'] = hist['Date'].dt.tz_localize(None)
                
            hist.insert(0, 'Company Name', company_name)
            hist.insert(1, 'PSX Ticker', ticker.replace(".KA", ""))
            
            cols_to_keep = ['Company Name', 'PSX Ticker', 'Date', 'Open', 'High', 'Low', 'Close', 'Adj Close', 'Volume']
            hist = hist[[c for c in cols_to_keep if c in hist.columns]]
            all_dataframes.append(hist)
            
        except Exception as e:
            print(f"    -> Error processing {ticker}: {e}")

    if all_dataframes:
        master_df = pd.concat(all_dataframes, ignore_index=True)
        master_df = master_df.sort_values(by=['Company Name', 'Date'])
        output_file = "PSX_77Firms_Daily_MarketData.xlsx"
        master_df.to_excel(output_file, index=False)
        print(f"\nSuccess! Data saved to {output_file}")

if __name__ == "__main__":
    main()

Connecting to Yahoo Finance to download daily data for 77 companies...
[1/77] Fetching IDYM.KA (Indus Dyeing & Manufacturing Ltd)...
[2/77] Fetching STJT.KA (Shahtaj Textile Ltd)...
[3/77] Fetching ANL.KA (Azgard Nine Ltd)...
[4/77] Fetching FML.KA (Feroze 1888 Mills Ltd)...
[5/77] Fetching GATM.KA (Gul Ahmed Textile Mills Ltd)...
[6/77] Fetching KTML.KA (Kohinoor Textile Mills Ltd)...
[7/77] Fetching NCL.KA (Nishat (Chunian) Ltd)...
[8/77] Fetching NML.KA (Nishat Mills Ltd)...
[9/77] Fetching BNWM.KA (Bannu Woollen Mills Ltd)...
[10/77] Fetching IBFL.KA (Ibrahim Fibres Ltd)...
[11/77] Fetching JDWS.KA (JDW Sugar Mills Ltd)...
[12/77] Fetching BWCL.KA (Bestway Cement Ltd)...
[13/77] Fetching CHCC.KA (Cherat Cement Company Ltd)...
[14/77] Fetching DGKC.KA (D. G. Khan Cement Co. Ltd)...
[15/77] Fetching FCCL.KA (Fauji Cement Company Ltd.)...
[16/77] Fetching KOHC.KA (Kohat Cement Company Ltd.)...
[17/77] Fetching LUCK.KA (Lucky Cement Ltd.)...
[18/77] Fetching MLCF.KA (Maple Leaf Cement 

$AGP.KA: possibly delisted; no price data found  (1d 2020-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1577818800, endDate = 1735585200")


   -> WARNING: No data for AGP. Please place 'AGP_Manual_Data.csv' in this folder.
[60/77] Fetching GLAXO.KA (GlaxoSmithKline Pakistan Ltd)...
[61/77] Fetching HALEON.KA (GlaxoSmithKline Consumer HealthCare)...
[62/77] Fetching SEARL.KA (The Searle Company Ltd)...
[63/77] Fetching ARPL.KA (Archroma Pakistan Ltd)...
[64/77] Fetching COLG.KA (Colgate-Palmolive (Pakistan) Ltd)...
[65/77] Fetching EPCL.KA (Engro Polymer & Chemicals Ltd)...
[66/77] Fetching LCI.KA (I. C. I. Pakistan Ltd)...
[67/77] Fetching LOTCHEM.KA (Lotte Chemical Pakistan Ltd)...
[68/77] Fetching PKGS.KA (Packages Ltd)...
[69/77] Fetching UNITY.KA (Unity Foods Ltd)...
[70/77] Fetching SRVI.KA (Service Industries Ltd)...
[71/77] Fetching FCEPL.KA (Engro Foods Ltd)...
[72/77] Fetching MUREB.KA (Murree Brewery Company Ltd)...
[73/77] Fetching NESTLE.KA (Nestle Pakistan Ltd)...
[74/77] Fetching NATF.KA (National Foods Ltd)...
[75/77] Fetching GHGL.KA (Ghani Glass Ltd)...
[76/77] Fetching PSEL.KA (Pakistan Services Ltd)...
[

In [25]:
import numpy as np
import os

warnings.filterwarnings("ignore")

# Exact mapping (Order preserved)
company_ticker_map = {
    "Indus Dyeing & Manufacturing Ltd": "IDYM.KA",
    "Shahtaj Textile Ltd": "STJT.KA",
    "Azgard Nine Ltd": "ANL.KA",
    "Feroze 1888 Mills Ltd": "FML.KA",
    "Gul Ahmed Textile Mills Ltd": "GATM.KA",
    "Kohinoor Textile Mills Ltd": "KTML.KA",
    "Nishat (Chunian) Ltd": "NCL.KA",
    "Nishat Mills Ltd": "NML.KA",
    "Bannu Woollen Mills Ltd": "BNWM.KA",
    "Ibrahim Fibres Ltd": "IBFL.KA",
    "JDW Sugar Mills Ltd": "JDWS.KA",
    "Bestway Cement Ltd": "BWCL.KA",
    "Cherat Cement Company Ltd": "CHCC.KA",
    "D. G. Khan Cement Co. Ltd": "DGKC.KA",
    "Fauji Cement Company Ltd.": "FCCL.KA",
    "Kohat Cement Company Ltd.": "KOHC.KA",
    "Lucky Cement Ltd.": "LUCK.KA",
    "Maple Leaf Cement Factory Ltd": "MLCF.KA",
    "Pioneer Cement Ltd": "PIOC.KA",
    "Pakistan Tobacco Company Ltd.": "PAKT.KA",
    "Philip Morris (Pakistan) Ltd": "PMPK.KA",
    "Attock Refinery Ltd": "ATRL.KA",
    "Byco Petroleum Pakistan Ltd": "CNERGY.KA",
    "K-Electric Ltd": "KEL.KA",
    "Kot Addu Power Company Ltd": "KAPCO.KA",
    "Saif Power Ltd": "SPWL.KA",
    "The Hub Power Company Ltd": "HUBC.KA",
    "Nishat Chunian Power Ltd": "NCPL.KA",
    "Attock Petroleum Ltd": "APL.KA",
    "Hascol Petroleum Ltd": "HASCOL.KA",
    "Pakistan State Oil Company Ltd": "PSO.KA",
    "Shell Pakistan Ltd": "SHEL.KA",
    "Sui Northern Gas Pipelines Ltd.": "SNGP.KA",
    "Sui Southern Gas Co. Ltd": "SSGC.KA",
    "Mari Petroleum Company Ltd": "MARI.KA",
    "Oil & Gas Development Co. Ltd.": "OGDC.KA",
    "Pakistan Oilfields Ltd": "POL.KA",
    "Pakistan Petroleum Ltd": "PPL.KA",
    "International Industries Ltd": "INIL.KA",
    "International Steels Ltd": "ISL.KA",
    "Atlas Honda Ltd": "ATLH.KA",
    "Honda Atlas Cars (Pakistan) Ltd": "HCAR.KA",
    "Indus Motor Company Ltd": "INDU.KA",
    "Millat Tractors Ltd": "MTL.KA",
    "Pak Suzuki Motor Co. Ltd.": "PSMC.KA",
    "Agriauto Industries Ltd": "AGIL.KA",
    "Thal Ltd": "THALL.KA",
    "Pak Elektron Ltd": "PAEL.KA",
    "Pakistan International Bulk Term. Ltd.": "PIBTL.KA",
    "P. T. C. L. \"A\"": "PTC.KA",
    "Systems Ltd": "SYS.KA",
    "TRG Pakistan Ltd": "TRG.KA",
    "Engro Corporation Ltd": "ENGRO.KA",
    "Engro Fertilizers Ltd": "EFERT.KA",
    "Fatima Fertilizer Company Ltd": "FATIMA.KA",
    "Fauji Fertilizer Bin Qasim Ltd": "FFBL.KA",
    "Fauji Fertilizer Company Ltd": "FFC.KA",
    "Abbott Laboratories (Pak.) Ltd": "ABOT.KA",
    "AGP Ltd": "AGP.KA",
    "GlaxoSmithKline Pakistan Ltd": "GLAXO.KA",
    "GlaxoSmithKline Consumer HealthCare": "HALEON.KA",
    "The Searle Company Ltd": "SEARL.KA",
    "Archroma Pakistan Ltd": "ARPL.KA",
    "Colgate-Palmolive (Pakistan) Ltd": "COLG.KA",
    "Engro Polymer & Chemicals Ltd": "EPCL.KA",
    "I. C. I. Pakistan Ltd": "LCI.KA",
    "Lotte Chemical Pakistan Ltd": "LOTCHEM.KA",
    "Packages Ltd": "PKGS.KA",
    "Unity Foods Ltd": "UNITY.KA",
    "Service Industries Ltd": "SRVI.KA",
    "Engro Foods Ltd": "FCEPL.KA",
    "Murree Brewery Company Ltd": "MUREB.KA",
    "Nestle Pakistan Ltd": "NESTLE.KA",
    "National Foods Ltd": "NATF.KA",
    "Ghani Glass Ltd": "GHGL.KA",
    "Pakistan Services Ltd": "PSEL.KA",
    "Shifa International Hospitals Ltd": "SHFA.KA"
}

def main():
    start_date = "2020-01-01"
    end_date = "2024-12-31"
    manual_agp_file = "AGP_Manual_Data.csv"
    
    yearly_results = []

    for name, ticker in company_ticker_map.items():
        print(f"Processing {name} ({ticker})...")
        try:
            stock = yf.Ticker(ticker)
            df = stock.history(start=start_date, end=end_date, auto_adjust=False)
            
            # AGP Fallback logic
            if df.empty and ticker == "AGP.KA" and os.path.exists(manual_agp_file):
                df = pd.read_csv(manual_agp_file)
                df['Date'] = pd.to_datetime(df['Date'])
                df.set_index('Date', inplace=True)
                df = df.rename(columns={'Price': 'Close', 'Vol.': 'Volume'})

            if df.empty:
                continue

            # Calculate Daily Returns for Volatility
            df['Daily_Return'] = df['Adj Close'].pct_change()
            df['Log_Return'] = np.log(df['Adj Close'] / df['Adj Close'].shift(1))

            # Group by Year
            df['Year'] = df.index.year
            
            for year, group in df.groupby('Year'):
                # Stock Return: (End of Year Price / Start of Year Price) - 1
                start_price = group['Adj Close'].iloc[0]
                end_price = group['Adj Close'].iloc[-1]
                yearly_return = (end_price / start_price) - 1
                
                # Volatility: Std Dev of Log Returns * sqrt(252)
                yearly_vol = group['Log_Return'].std() * np.sqrt(252)
                
                # Trading Volume: Sum of daily volume
                total_volume = group['Volume'].sum()

                yearly_results.append({
                    "Company Name": name,
                    "Ticker": ticker.replace(".KA", ""),
                    "Year": year,
                    "Stock_Return": yearly_return,
                    "Stock_Volatility": yearly_vol,
                    "Trading_Volume": total_volume
                })

        except Exception as e:
            print(f"Error with {ticker}: {e}")

    # Create DataFrame and Export
    final_df = pd.DataFrame(yearly_results)
    output_file = "PSX_Yearly_Market_Data_2020_2024.xlsx"
    final_df.to_excel(output_file, index=False)
    print(f"\nSuccess! Yearly data saved to {output_file}")

if __name__ == "__main__":
    main()

Processing Indus Dyeing & Manufacturing Ltd (IDYM.KA)...
Processing Shahtaj Textile Ltd (STJT.KA)...
Processing Azgard Nine Ltd (ANL.KA)...
Processing Feroze 1888 Mills Ltd (FML.KA)...
Processing Gul Ahmed Textile Mills Ltd (GATM.KA)...
Processing Kohinoor Textile Mills Ltd (KTML.KA)...
Processing Nishat (Chunian) Ltd (NCL.KA)...
Processing Nishat Mills Ltd (NML.KA)...
Processing Bannu Woollen Mills Ltd (BNWM.KA)...
Processing Ibrahim Fibres Ltd (IBFL.KA)...
Processing JDW Sugar Mills Ltd (JDWS.KA)...
Processing Bestway Cement Ltd (BWCL.KA)...
Processing Cherat Cement Company Ltd (CHCC.KA)...
Processing D. G. Khan Cement Co. Ltd (DGKC.KA)...
Processing Fauji Cement Company Ltd. (FCCL.KA)...
Processing Kohat Cement Company Ltd. (KOHC.KA)...
Processing Lucky Cement Ltd. (LUCK.KA)...
Processing Maple Leaf Cement Factory Ltd (MLCF.KA)...
Processing Pioneer Cement Ltd (PIOC.KA)...
Processing Pakistan Tobacco Company Ltd. (PAKT.KA)...
Processing Philip Morris (Pakistan) Ltd (PMPK.KA)...
Proc

$AGP.KA: possibly delisted; no price data found  (1d 2020-01-01 -> 2024-12-31) (Yahoo error = "Data doesn't exist for startDate = 1577818800, endDate = 1735585200")


Processing GlaxoSmithKline Pakistan Ltd (GLAXO.KA)...
Processing GlaxoSmithKline Consumer HealthCare (HALEON.KA)...
Processing The Searle Company Ltd (SEARL.KA)...
Processing Archroma Pakistan Ltd (ARPL.KA)...
Processing Colgate-Palmolive (Pakistan) Ltd (COLG.KA)...
Processing Engro Polymer & Chemicals Ltd (EPCL.KA)...
Processing I. C. I. Pakistan Ltd (LCI.KA)...
Processing Lotte Chemical Pakistan Ltd (LOTCHEM.KA)...
Processing Packages Ltd (PKGS.KA)...
Processing Unity Foods Ltd (UNITY.KA)...
Processing Service Industries Ltd (SRVI.KA)...
Processing Engro Foods Ltd (FCEPL.KA)...
Processing Murree Brewery Company Ltd (MUREB.KA)...
Processing Nestle Pakistan Ltd (NESTLE.KA)...
Processing National Foods Ltd (NATF.KA)...
Processing Ghani Glass Ltd (GHGL.KA)...
Processing Pakistan Services Ltd (PSEL.KA)...
Processing Shifa International Hospitals Ltd (SHFA.KA)...

Success! Yearly data saved to PSX_Yearly_Market_Data_2020_2024.xlsx
